# DP2 — Tracts couvrant les Deep Drilling Fields (DDF)

**Auteur :** dagoret  
**Date :** 2026-03-13  

Ce notebook :
1. Instancie un Butler Gen3 sur le dépôt DP0.2
2. Charge le SkyMap et récupère tous les tracts disponibles
3. Identifie les tracts qui recouvrent les champs DDF (WFD et uDDF de DC2 / DDFs LSST standards)
4. Affiche en projection Mollweide (astropy WCSAxes) les patchs colorés par tract, avec les centres DDF

**Référence DDF :** `2026-03-11_DP2_SurveyPropertyMaps_AllBands_AllDDF.ipynb`

## 0. Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.path import Path
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.collections import PatchCollection
import seaborn as sns

# Astropy
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.wcs import WCS

# LSST Stack
import lsst.daf.butler as dafButler
import lsst.geom as geom
import lsst.sphgeom as sphgeom

print(f"Matplotlib version : {matplotlib.__version__}")
print(f"Seaborn version    : {sns.__version__}")
print("Imports OK")

## 1. Définition des Deep Drilling Fields

Coordonnées issues de `2026-03-11_DP2_SurveyPropertyMaps_AllBands_AllDDF.ipynb`.

Pour DP0.2 (basé sur DC2) la simulation couvre ~300 deg² centrée autour de RA≈61.9°, Dec≈-35.8°.  
Le champ ultra-profond (uDDF) de DC2 est centré sur RA=55.064°, Dec=-29.783°.  
Les champs DDF LSST standards sont également listés ci-dessous.

In [ ]:
# -------------------------------------------------------------------------
# Dictionnaire des DDF : nom -> (RA_deg, Dec_deg)
# Source : notebook de référence 2026-03-11 + littérature LSST
# -------------------------------------------------------------------------
DDF_COORDS = {
    # --- Champs présents dans DC2 / DP0.2 ---
    #"DC2_WFD_center" : (61.863, -35.790),   # Centre de la simulation WFD DC2
    #"DC2_uDDF"       : (55.064, -29.783),   # Ultra-deep drilling field DC2

    # --- DDF LSST officiels (hors empreinte DC2, pour référence) ---
    "COSMOS"         : (150.119, +2.206),
    "ECDFS"          : (53.125, -28.100),   # Extended Chandra Deep Field South
    "ELAIS-S1"       : (9.450,  -44.000),
    "XMM-LSS"        : (35.708, -4.750),
    "EDFS-a"         : (58.900, -49.315),   # Euclid Deep Field South (a)
    "EDFS-b"         : (63.600, -47.600),   # Euclid Deep Field South (b)
    "EDFS"           : (61.24,-48.423),
    "M49"            : (187.4 ,8.0),
}

# Rayon de recherche autour de chaque DDF (en degrés)
# Un tract HSC-like fait ~1.7 deg de côté → rayon de 1.5 deg capture le(s) tract(s) central/aux
SEARCH_RADIUS_DEG = 1.8

print("Champs DDF définis :")
for name, (ra, dec) in DDF_COORDS.items():
    print(f"  {name:20s}  RA={ra:8.3f}°   Dec={dec:+8.3f}°")

## 2. Instanciation du Butler et chargement du SkyMap

In [ ]:
# -------------------------------------------------------------------------
# Configuration Butler DP0.2
# Adapter le chemin REPO selon l'environnement (RSP, NERSC, local…)
# -------------------------------------------------------------------------
REPO       = "dp2_prep"                          # alias RSP
COLLECTION = "LSSTCam/runs/DRP/20250417_20250921/w_2025_49/DM-53545"
SKYMAP_NAME = "lsst_cells_v2"   


butler  = dafButler.Butler(REPO, collections=COLLECTION)
skymap  = butler.get("skyMap", skymap=SKYMAP_NAME)

n_tracts = len(skymap)
print(f"Butler instancié  : {REPO}")
print(f"Collection        : {COLLECTION}")
print(f"SkyMap            : {SKYMAP_NAME}")
print(f"Nombre de tracts  : {n_tracts}")

## 3. Recherche des tracts qui recouvrent les DDF

In [ ]:
# -------------------------------------------------------------------------
# Fonction : retourne tous les tracts recouvrant un disque (ra, dec, radius)
#
# Approche : grille de points échantillonnant le disque de recherche.
# Pour chaque point on appelle skymap.findTract() qui renvoie toujours
# un TractInfo (méthode stable dans toutes les versions du stack LSST).
# On fait l'union des IDs trouvés → dédupliqué.
#
# Cette approche évite entièrement sphgeom.Circle / poly.intersects()
# dont le type de retour (bool vs Relationship) varie selon la version.
# -------------------------------------------------------------------------
def find_tracts_for_coord(skymap, ra_deg, dec_deg, radius_deg=SEARCH_RADIUS_DEG):
    """
    Retourne la liste dédupliquée des TractInfo dont l'empreinte recouvre
    le disque (ra_deg, dec_deg, radius_deg).
    Utilise uniquement skymap.findTract(SpherePoint) — API stable.
    """
    cos_dec = max(np.cos(np.deg2rad(dec_deg)), 0.01)   # éviter division par 0 aux pôles
    step    = 0.35   # pas de grille en degrés (< taille d'un tract)

    found_ids = set()

    # Grille rectangulaire couvrant le disque
    for ddec in np.arange(-radius_deg, radius_deg + step, step):
        for dra in np.arange(-radius_deg, radius_deg + step, step):
            # Distance angulaire au centre (approximation sphérique locale)
            if np.sqrt(dra**2 + ddec**2) > radius_deg:
                continue
            ra_s  = ra_deg  + dra / cos_dec    # correction de projection
            dec_s = dec_deg + ddec
            if not (-89.9 <= dec_s <= 89.9):
                continue
            sp = geom.SpherePoint(ra_s * geom.degrees, dec_s * geom.degrees)
            try:
                ti = skymap.findTract(sp)
                found_ids.add(ti.tract_id)
            except Exception:
                pass

    return [skymap[tid] for tid in sorted(found_ids)]


# -------------------------------------------------------------------------
# Recherche pour chaque DDF
# -------------------------------------------------------------------------
# ddf_tracts : dict  ddf_name -> list of TractInfo
ddf_tracts = {}

for ddf_name, (ra, dec) in DDF_COORDS.items():
    tracts = find_tracts_for_coord(skymap, ra, dec, radius_deg=SEARCH_RADIUS_DEG)
    ddf_tracts[ddf_name] = tracts
    ids = [t.tract_id for t in tracts]
    print(f"{ddf_name:20s}  → {len(tracts)} tract(s) : {ids}")

In [ ]:
# -------------------------------------------------------------------------
# Construire la liste dédupliquée de tous les tracts sélectionnés
# avec leur(s) association(s) DDF
# -------------------------------------------------------------------------
tract_to_ddfs = {}   # tract_id -> [ddf_name, ...]

for ddf_name, tracts in ddf_tracts.items():
    for t in tracts:
        tid = t.tract_id
        tract_to_ddfs.setdefault(tid, [])
        if ddf_name not in tract_to_ddfs[tid]:
            tract_to_ddfs[tid].append(ddf_name)

# Index unique des TractInfo sélectionnés
selected_tract_ids   = sorted(tract_to_ddfs.keys())
selected_tract_infos = {tid: skymap[tid] for tid in selected_tract_ids}

print(f"\nTotal tracts sélectionnés (dédupliqués) : {len(selected_tract_ids)}")
print(f"Tract IDs : {selected_tract_ids}")

## 4. Utilitaires de géométrie : extraction des coins des patchs

Pour chaque tract sélectionné on extrait les coordonnées RA/Dec des coins de chaque patch (inner bbox → WCS → sky coords).

In [ ]:
def get_tract_corners_radec(tract_info):
    """
    Retourne les 4 coins du tract (outer sky polygon) en (RA, Dec) degrés.
    L'ordre est maintenu pour former un polygone fermé.
    """
    vertices = tract_info.getOuterSkyPolygon().getVertices()
    corners = []
    for v in vertices:
        sp = geom.SpherePoint(v)
        corners.append((sp.getRa().asDegrees(), sp.getDec().asDegrees()))
    return np.array(corners)


def get_patch_corners_radec(tract_info, patch_info):
    """
    Retourne les 4 coins d'un patch (inner bbox) en coordonnées RA/Dec degrés.
    """
    wcs  = tract_info.getWcs()
    bbox = patch_info.getInnerBBox()

    pixel_corners = [
        geom.Point2D(bbox.getMinX(), bbox.getMinY()),
        geom.Point2D(bbox.getMaxX(), bbox.getMinY()),
        geom.Point2D(bbox.getMaxX(), bbox.getMaxY()),
        geom.Point2D(bbox.getMinX(), bbox.getMaxY()),
    ]
    radec = []
    for pix in pixel_corners:
        sp = wcs.pixelToSky(pix)
        radec.append((sp.getRa().asDegrees(), sp.getDec().asDegrees()))
    return np.array(radec)


def wrap_ra(ra_arr, ref_ra=180.0):
    """
    Ramène les RA dans [-180, 180] pour la projection Mollweide d'astropy.
    ra_arr en degrés → retour en degrés centré autour de ref_ra.
    """
    ra = np.array(ra_arr, dtype=float)
    ra = np.where(ra > 180.0, ra - 360.0, ra)
    return ra


print("Fonctions utilitaires définies.")

In [ ]:
# -------------------------------------------------------------------------
# Pré-calcul de tous les polygones de patchs pour les tracts sélectionnés
# -------------------------------------------------------------------------
# Structure : patch_polygons[tract_id] = list of (ra_arr, dec_arr)  [5 pts, polygone fermé]

patch_polygons = {}   # tract_id -> list of np.ndarray shape (5, 2) = [(ra, dec), ...]
tract_centers  = {}   # tract_id -> (ra_center, dec_center)

for tid in selected_tract_ids:
    ti = selected_tract_infos[tid]
    polys = []
    for patch_info in ti:
        corners = get_patch_corners_radec(ti, patch_info)   # shape (4, 2)
        # Fermer le polygone
        closed  = np.vstack([corners, corners[0]])           # shape (5, 2)
        polys.append(closed)

    patch_polygons[tid] = polys

    # Centre du tract (barycentre des coins du tract)
    tc = get_tract_corners_radec(ti)
    ra_c  = np.mean(tc[:, 0])
    dec_c = np.mean(tc[:, 1])
    tract_centers[tid] = (ra_c, dec_c)

    n_patches = sum(1 for _ in ti)
    print(f"Tract {tid:5d}  centre=({ra_c:.2f}°, {dec_c:+.2f}°)  "
          f"patchs={n_patches}  DDFs={tract_to_ddfs[tid]}")

## 5. Visualisation — Projection Mollweide (astropy WCSAxes)

- Les **patchs** sont colorés par tract, avec une couleur distincte par tract.
- Les **centres des DDF** sont marqués par une étoile.
- La **légende** est placée à gauche, hors de l'image.
- Palette seaborn pour des couleurs bien distinctes.

In [ ]:
# =========================================================================
# Paramètres graphiques
# =========================================================================

# --- Palette de couleurs seaborn bien distinctes ---
n_tracts_sel = len(selected_tract_ids)

# Pour ≤ 10 tracts : palette qualitative ; au-delà on utilise hls
if n_tracts_sel <= 10:
    palette = sns.color_palette("tab10", n_tracts_sel)
elif n_tracts_sel <= 20:
    palette = sns.color_palette("tab20", n_tracts_sel)
else:
    palette = sns.color_palette("hls", n_tracts_sel)

tract_color = {tid: palette[i] for i, tid in enumerate(selected_tract_ids)}

# --- Symboles et couleurs pour les centres DDF ---
DDF_MARKER      = "*"
DDF_MARKERSIZE  = 18
DDF_EDGECOLOR   = "black"
DDF_FACECOLOR   = "gold"

# =========================================================================
# Création de la figure Mollweide avec matplotlib
# Note : matplotlib supporte nativement la projection 'mollweide'
#        avec des angles en RADIANS et l'axe x en [-π, π]
# =========================================================================

fig = plt.figure(figsize=(18, 9))

# Axes Mollweide (matplotlib built-in — gère la projection sphérique)
# On réserve de la place à gauche pour la légende
ax = fig.add_subplot(111, projection='mollweide')
ax.set_facecolor('#f0f4f8')   # fond légèrement grisé

ax.grid(True, color='white', linewidth=0.6, alpha=0.8)

# =========================================================================
# Tracé des patchs colorés par tract
# =========================================================================

legend_handles = []

for tid in selected_tract_ids:
    color   = tract_color[tid]
    ddfs    = tract_to_ddfs[tid]
    label   = f"Tract {tid} :: " + ", ".join(ddfs)

    patch_added = False
    for poly in patch_polygons[tid]:
        ra_deg  = poly[:, 0]   # (5,)
        dec_deg = poly[:, 1]   # (5,)

        # Conversion en radians pour matplotlib Mollweide
        # RA : convention astropy Mollweide → l'est à gauche, donc -RA
        ra_rad  = -np.deg2rad(wrap_ra(ra_deg))
        dec_rad =  np.deg2rad(dec_deg)

        # Vérifier qu'il n'y a pas de discontinuité de phase (wrap)
        # (cas des polygones qui traversent RA=0/360 — absent dans DC2)
        if np.any(np.abs(np.diff(ra_rad)) > np.pi / 2):
            continue   # polygone coupé par le bord — ignorer pour simplifier

        verts  = np.column_stack([ra_rad, dec_rad])
        patch  = mpatches.Polygon(
            verts[:-1],          # sans le point de fermeture
            closed=True,
            facecolor=color,
            edgecolor='white',
            linewidth=0.3,
            alpha=0.65,
            transform=ax.transData,
        )
        ax.add_patch(patch)
        patch_added = True

    if patch_added:
        legend_handles.append(
            mpatches.Patch(facecolor=color, edgecolor='grey',
                           linewidth=0.8, alpha=0.85, label=label)
        )

# =========================================================================
# Tracé des contours de tracts (bord épais)
# =========================================================================

for tid in selected_tract_ids:
    color = tract_color[tid]
    ti    = selected_tract_infos[tid]
    tc    = get_tract_corners_radec(ti)

    ra_wrap = wrap_ra(tc[:, 0])
    ra_rad  = -np.deg2rad(np.append(ra_wrap, ra_wrap[0]))
    dec_rad =  np.deg2rad(np.append(tc[:, 1], tc[0, 1]))

    ax.plot(ra_rad, dec_rad,
            color=color, linewidth=1.8,
            solid_capstyle='round', zorder=3,
            path_effects=[pe.Stroke(linewidth=3.5, foreground='black'), pe.Normal()])

# =========================================================================
# Tracé des centres DDF
# =========================================================================

ddf_handles = []
for ddf_name, (ra, dec) in DDF_COORDS.items():
    ra_rad  = -np.deg2rad(wrap_ra(ra))
    dec_rad =  np.deg2rad(dec)
    ax.plot(ra_rad, dec_rad,
            marker=DDF_MARKER,
            markersize=DDF_MARKERSIZE,
            color=DDF_FACECOLOR,
            markeredgecolor=DDF_EDGECOLOR,
            markeredgewidth=1.0,
            zorder=10,
            linestyle='none')
    # Annotation
    ax.annotate(
        ddf_name,
        xy=(ra_rad, dec_rad),
        xytext=(ra_rad + np.deg2rad(2.0), dec_rad + np.deg2rad(1.5)),
        fontsize=7,
        fontweight='bold',
        color='black',
        arrowprops=dict(arrowstyle='->', color='black', lw=0.8),
        zorder=11,
    )

ddf_handles.append(
    plt.Line2D([0], [0],
               marker=DDF_MARKER, linestyle='none',
               markersize=12,
               color=DDF_FACECOLOR,
               markeredgecolor=DDF_EDGECOLOR,
               markeredgewidth=1.0,
               label='Centre DDF')
)

# =========================================================================
# Légende — à l'extérieur à gauche
# =========================================================================

all_handles = legend_handles + ddf_handles

leg = ax.legend(
    handles=all_handles,
    loc='center left',
    bbox_to_anchor=(-0.30, 0.50),
    fontsize=8,
    frameon=True,
    framealpha=0.92,
    edgecolor='grey',
    title='Tract :: DDF',
    title_fontsize=9,
    ncol=1,
)

# =========================================================================
# Axes et titre
# =========================================================================

# Ticks RA en heures (matplotlib Mollweide met en radians)
tick_ra_deg  = np.arange(0, 360, 30)
tick_ra_rad  = -np.deg2rad(wrap_ra(tick_ra_deg))
ax.set_xticks(tick_ra_rad)
ax.set_xticklabels(
    [f"{r:.0f}°" for r in tick_ra_deg],
    fontsize=8
)

tick_dec_deg = np.arange(-75, 90, 15)
ax.set_yticks(np.deg2rad(tick_dec_deg))
ax.set_yticklabels([f"{d:+.0f}°" for d in tick_dec_deg], fontsize=8)

ax.set_xlabel("Ascension Droite (J2000)", fontsize=11, labelpad=12)
ax.set_ylabel("Déclinaison (J2000)", fontsize=11)

ax.set_title(
    "DP0.2 — Tracts du SkyMap DC2 recouvrant les Deep Drilling Fields\n"
    "(Projection Mollweide — patchs colorés par tract)",
    fontsize=13, fontweight='bold', pad=14
)

plt.tight_layout()
fig.subplots_adjust(left=0.28)   # espace pour la légende à gauche

# Sauvegarde
outfile = "DDF_Tracts_Mollweide_DP02.png"
plt.savefig(outfile, dpi=180, bbox_inches='tight')
print(f"Figure sauvegardée : {outfile}")
plt.show()

## 6. Tableau récapitulatif

In [ ]:
import pandas as pd

rows = []
for tid in selected_tract_ids:
    ra_c, dec_c = tract_centers[tid]
    ddfs        = ", ".join(tract_to_ddfs[tid])
    n_p         = len(patch_polygons[tid])
    rows.append({
        "tract_id"    : tid,
        "RA_center"   : round(ra_c, 3),
        "Dec_center"  : round(dec_c, 3),
        "n_patches"   : n_p,
        "DDF(s)"      : ddfs,
    })

df = pd.DataFrame(rows).set_index("tract_id")
print(f"Tracts sélectionnés ({len(df)} au total) :")
display(df)

## 7. Zoom sur la région DC2 (RA ∈ [45°, 75°], Dec ∈ [−40°, −20°])

Projection gnomonique centrée sur la simulation DC2 pour un meilleur aperçu des tracts WFD et uDDF.

In [ ]:
# =========================================================================
# Zoom : vue cartésienne (RA/Dec) de la région DC2
# =========================================================================

RA_MIN,  RA_MAX  = 40.0,  80.0
DEC_MIN, DEC_MAX = -45.0, -20.0

fig2, ax2 = plt.subplots(figsize=(14, 8))
ax2.set_facecolor('#f0f4f8')
ax2.invert_xaxis()   # convention : RA croissant vers la gauche

legend_handles2 = []
for tid in selected_tract_ids:
    color = tract_color[tid]
    ddfs  = tract_to_ddfs[tid]
    label = f"Tract {tid} :: " + ", ".join(ddfs)

    # N'afficher que les tracts dans la fenêtre DC2
    ra_c, dec_c = tract_centers[tid]
    if not (RA_MIN <= ra_c <= RA_MAX and DEC_MIN <= dec_c <= DEC_MAX):
        continue

    patch_added = False
    for poly in patch_polygons[tid]:
        ra_deg  = poly[:-1, 0]
        dec_deg = poly[:-1, 1]
        p = mpatches.Polygon(
            list(zip(ra_deg, dec_deg)),
            closed=True,
            facecolor=color, edgecolor='white',
            linewidth=0.2, alpha=0.6,
        )
        ax2.add_patch(p)
        patch_added = True

    if patch_added:
        # Contour du tract
        ti = selected_tract_infos[tid]
        tc = get_tract_corners_radec(ti)
        ax2.plot(np.append(tc[:, 0], tc[0, 0]),
                 np.append(tc[:, 1], tc[0, 1]),
                 color=color, linewidth=2.0, zorder=3,
                 path_effects=[pe.Stroke(linewidth=3.5, foreground='black'), pe.Normal()])
        legend_handles2.append(
            mpatches.Patch(facecolor=color, edgecolor='grey',
                           linewidth=0.8, alpha=0.85, label=label)
        )

# Centres DDF dans la fenêtre
for ddf_name, (ra, dec) in DDF_COORDS.items():
    if RA_MIN <= ra <= RA_MAX and DEC_MIN <= dec <= DEC_MAX:
        ax2.plot(ra, dec, marker=DDF_MARKER, markersize=16,
                 color=DDF_FACECOLOR, markeredgecolor=DDF_EDGECOLOR,
                 markeredgewidth=1.2, zorder=10, linestyle='none')
        ax2.annotate(ddf_name, xy=(ra, dec),
                     xytext=(ra + 0.8, dec + 0.8),
                     fontsize=9, fontweight='bold',
                     arrowprops=dict(arrowstyle='->', color='black', lw=0.7))

legend_handles2.append(
    plt.Line2D([0], [0], marker=DDF_MARKER, linestyle='none',
               markersize=12, color=DDF_FACECOLOR,
               markeredgecolor=DDF_EDGECOLOR, markeredgewidth=1.0,
               label='Centre DDF')
)

ax2.set_xlim(RA_MAX, RA_MIN)
ax2.set_ylim(DEC_MIN, DEC_MAX)
ax2.set_xlabel("Ascension Droite (deg)", fontsize=12)
ax2.set_ylabel("Déclinaison (deg)", fontsize=12)
ax2.set_title(
    "DP0.2 — Zoom sur la région DC2 : tracts et DDF\n"
    f"(RA ∈ [{RA_MIN}°, {RA_MAX}°]  Dec ∈ [{DEC_MIN}°, {DEC_MAX}°])",
    fontsize=13, fontweight='bold'
)
ax2.grid(True, alpha=0.4, linestyle='--')

ax2.legend(
    handles=legend_handles2,
    loc='center left',
    bbox_to_anchor=(-0.30, 0.50),
    fontsize=8.5,
    frameon=True, framealpha=0.92, edgecolor='grey',
    title='Tract :: DDF', title_fontsize=9,
)

plt.tight_layout()
fig2.subplots_adjust(left=0.28)

outfile2 = "DDF_Tracts_ZoomDC2_DP02.png"
plt.savefig(outfile2, dpi=180, bbox_inches='tight')
print(f"Figure sauvegardée : {outfile2}")
plt.show()

---
## Notes techniques

| Paramètre | Valeur |
|---|---|
| Dépôt Butler | `dp02` |
| Collection | `2.2i/runs/DP0.2` |
| SkyMap | `DC2` (7×7 patchs/tract, 0.2"/pix) |
| Rayon de recherche | 1.8° autour de chaque centre DDF |
| Projection globale | Mollweide (matplotlib natif) |
| Projection locale | Cartésienne (RA/Dec direct) |
| Palette couleurs | `seaborn tab10/tab20/hls` selon le nombre de tracts |

**Adapter** `REPO`, `COLLECTION` et `SKYMAP_NAME` si vous travaillez sur un environnement différent du RSP (NERSC, local stack, etc.).